In [1]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_score, recall_score, f1_score,
                             precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')

print(f"Dataset: {len(ind_hosp)} rows, {len(ind_hosp.columns)} columns")
print(f"Columns: {ind_hosp.columns.tolist()}")

Dataset: 1823640 rows, 138 columns
Columns: ['subject_id', 'hadm_id', 'dischtime', 'los', 'age', 'day', 'current_date', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025', 'ccsr_END011', 'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007', 'ccsr_END003', 'ccsr_CIR019', 'ccsr_DIG004', 'ccsr_CIR017', 'ccsr_CIR008', 'ccsr_BLD003', 'ccsr_EXT027', 'ccsr_GEN002', 'ccsr_END009', 'icd_I10', 'icd_E785', 'icd_K219', 'icd_Z87891', 'icd_I2510', 'icd_N179', 'icd_F329', 'icd_I4891', 'icd_Z7901', 'icd_F419', 'icd_E119', 'icd_E039', 'icd_Z794', 'icd_D649', 'icd_N390', 'comorbidity_score', 'num_procedures_daily', 'cumulative_procedures', 'num_medications_daily', 'cumulative_medications', 'lab_50868_daily', 'lab_50882_daily', 'lab_50893_daily', 'lab_50902_daily', 'lab_50912_daily', 'lab_50931_daily', 'lab_50960_daily', 'lab_50970_daily', 'lab_50971_daily', 'lab_50983_daily', 'lab_51006_daily', 'lab_51221_daily', 'lab_51222_daily', 'lab_51248_daily', 'lab_51249_daily', 'lab_51250_daily', 'lab_51265_dai

In [2]:
cols_to_drop = [
    'subject_id', 
    'hadm_id', 
    'dischtime', 
    'current_date',
    'target_readmission_30d',
    'los'
]
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]
X = ind_hosp.drop(columns=cols_to_drop)
y = ind_hosp['target_readmission_30d']

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

print(f"\nFeatures: {X.shape[1]}, target: {y.name}")
print(f"Readmission rate: {y.mean():.2%}")

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.125,
    random_state=42,
    stratify=patient_target[patient_target['subject_id'].isin(train_val_ids)]['has_readmission']
)

train_mask = ind_hosp['subject_id'].isin(train_ids)
val_mask = ind_hosp['subject_id'].isin(val_ids)
test_mask = ind_hosp['subject_id'].isin(test_ids)

train_rate = ind_hosp[train_mask]['target_readmission_30d'].mean()
val_rate = ind_hosp[val_mask]['target_readmission_30d'].mean()
test_rate = ind_hosp[test_mask]['target_readmission_30d'].mean()

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

row_weights = 1 / ind_hosp.loc[train_mask, 'los']
row_weights = row_weights / row_weights.mean()

print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Val readmission rate: {y_val.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': 25,
    'min_child_samples': 20,
    'learning_rate': 0.02,
    'n_estimators': 1000,
    'subsample': 0.7,
    'subsample_freq': 1,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.3,
    'min_split_gain': 0.01,
    'scale_pos_weight': scale_pos_weight,
    'is_unbalance': False, 
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

model = lgb.LGBMClassifier(**params)

print("Training LightGBM...")
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    sample_weight=row_weights,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

calibrated = CalibratedClassifierCV(
    model,
    method='sigmoid',
    cv='prefit'
)
calibrated.fit(X_val, y_val)  # калибровка на валидационной выборке

y_val_proba = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
print(f"Optimal threshold: {best_threshold:.4f}")

y_pred_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= best_threshold).astype(int)

auc_roc = roc_auc_score(y_test, y_pred_proba)
auc_prc = average_precision_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\nAUC-ROC: {auc_roc:.4f}")
print(f"AUC-PRC: {auc_prc:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"\n(Threshold: {best_threshold:.4f}):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"NPV: {npv:.4f}")
print(f"F1-Score: {f1:.4f}")

importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop-20 (LightGBM):")
print(importances.head(100).to_string(index=False))


Features: 132, target: target_readmission_30d
Readmission rate: 20.07%
Train readmission rate: 20.01%
Val readmission rate: 20.41%
Test readmission rate: 20.13%
scale_pos_weight: 4.00
Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.695021
[100]	valid_0's auc: 0.698257
[150]	valid_0's auc: 0.70057
[200]	valid_0's auc: 0.702201
[250]	valid_0's auc: 0.703478
[300]	valid_0's auc: 0.704253
[350]	valid_0's auc: 0.704653
[400]	valid_0's auc: 0.704981
[450]	valid_0's auc: 0.705221
Early stopping, best iteration is:
[440]	valid_0's auc: 0.705281
Optimal threshold: 0.2158

AUC-ROC: 0.7001
AUC-PRC: 0.3850
Brier Score: 0.1463

(Threshold: 0.2158):
Accuracy: 0.6652
Precision: 0.3217
Recall: 0.5983
Specificity: 0.6820
NPV: 0.8708
F1-Score: 0.4184

Top-20 (LightGBM):
                                              feature  importance
                                                  age        1995
                                        num_diag

In [3]:
import joblib

model_path = 'lightgbm.pkl'
joblib.dump(calibrated, model_path)
print(f"\nModel saved to {model_path}")


Model saved to lightgbm.pkl
